# Building Player Career Stats from Play-by-Play Data

## Why are we doing this?
The database does not include a pre-built player stats table. Instead, it stores every individual play that happened in every game — 13.6 million rows in the `play_by_play` table.

We need to **aggregate those individual plays** up to per-player, per-season stats.

## What we will build
For each player in each season:
- Games played
- Field goals made
- Rebounds
- Assists
- Steals
- Blocks
- Turnovers

Then we roll those up to **career totals** per player.

## Approach
We use SQL to do the aggregation directly in the database rather than loading 13.6M rows into Python. This is much faster and more memory efficient.

In [1]:
import sqlite3
import pandas as pd

DB_PATH = r"E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\nba.sqlite"
conn = sqlite3.connect(DB_PATH)
print("Connected to database")

Connected to database


In [2]:
# Filter to players drafted in 2000 or later
# We create a temp table of valid player IDs once here and reuse it in all queries
conn.execute("DROP TABLE IF EXISTS temp_valid_players")
conn.execute("""
    CREATE TEMPORARY TABLE temp_valid_players AS
    SELECT DISTINCT person_id AS player_id
    FROM draft_history
    WHERE season >= 2000
""")
conn.commit()

valid_count = pd.read_sql_query("SELECT COUNT(*) as n FROM temp_valid_players", conn)
print(f"Players drafted 2000 or later: {valid_count['n'][0]}")

Players drafted 2000 or later: 1425


## Step 1 — Understand the Play-by-Play Structure

Each row in `play_by_play` represents one event in a game.

Key columns:
- `game_id` — links back to the `game` table to get the season
- `eventmsgtype` — a number code for what type of event happened (shot, foul, rebound etc)
- `player1_id` — the primary player involved (e.g. the shooter)
- `player2_id` — secondary player (e.g. the assister or blocker)
- `player3_id` — tertiary player (rare, used for some block/foul situations)
- `homedescription` / `visitordescription` — text description of the play

In [3]:
# Look at a sample of raw play-by-play rows
sample = pd.read_sql_query("""
    SELECT game_id, eventmsgtype, player1_id, player1_name,
           player2_id, player2_name,
           homedescription, visitordescription
    FROM play_by_play
    LIMIT 20
""", conn)
sample

,game_id,eventmsgtype,player1_id,player1_name,player2_id,player2_name,homedescription,visitordescription
0,0029600012,12,0,None,0,None,None,None
1,0029600012,10,406,Shaquille O'Neal,170,Joe Kleine,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None
2,0029600012,2,208,Sam Cassell,0,None,None,MISS Cassell 15' Jump Shot
3,0029600012,4,406,Shaquille O'Neal,0,None,O'Neal REBOUND (Off:0 Def:1),None
4,0029600012,2,76,Cedric Ceballos,0,None,MISS Ceballos 26' 3PT Jump Shot,None
5,0029600012,4,208,Sam Cassell,0,None,None,Cassell REBOUND (Off:0 Def:1)
6,0029600012,6,89,Nick Van Exel,0,None,Van Exel P.FOUL (P1.T1),None
7,0029600012,5,208,Sam Cassell,0,None,None,Cassell Bad Pass Turnover (P1.T1)
8,0029600012,2,76,Cedric Ceballos,0,None,MISS Ceballos 1' Layup,Horry BLOCK (1 BLK)
9,0029600012,4,1610612747,None,0,None,LAKERS Rebound,None


## Step 2 — Map Event Type Codes

The `eventmsgtype` column uses numeric codes. Here is what each code means:

| Code | Event |
|---|---|
| 1 | Made field goal (player1 scored, player2 assisted) |
| 2 | Missed field goal (player3 blocked if present) |
| 3 | Free throw |
| 4 | Rebound (player1 grabbed it) |
| 5 | Turnover (player1 turned it over, player2 stole it) |
| 6 | Foul |
| 7 | Violation |
| 8 | Substitution |
| 9 | Timeout |
| 10 | Jump ball |
| 12 | Period start |
| 13 | Period end |

In [4]:
# Verify the event type codes actually in the data
# and map them to human readable names
event_counts = pd.read_sql_query("""
    SELECT eventmsgtype, COUNT(*) as total_events
    FROM play_by_play
    GROUP BY eventmsgtype
    ORDER BY eventmsgtype
""", conn)

event_names = {
    1: 'Made Field Goal',
    2: 'Missed Field Goal',
    3: 'Free Throw',
    4: 'Rebound',
    5: 'Turnover',
    6: 'Foul',
    7: 'Violation',
    8: 'Substitution',
    9: 'Timeout',
    10: 'Jump Ball',
    12: 'Period Start',
    13: 'Period End'
}

event_counts['event_name'] = event_counts['eventmsgtype'].map(event_names)
event_counts

,eventmsgtype,total_events,event_name
0,1,2242357,Made Field Goal
1,2,2699376,Missed Field Goal
2,3,1442567,Free Throw
3,4,3049623,Rebound
4,5,868202,Turnover
5,6,1305746,Foul
6,7,53184,Violation
7,8,1226332,Substitution
8,9,380586,Timeout
9,10,52099,Jump Ball


## Step 3 — Link Games to Seasons

The `play_by_play` table has a `game_id` but no season. We join to the `game` table to get the `season_id`.

The `season_id` format is a 5-digit number like `22023` where:
- First digit = season type (2 = regular season, 4 = playoffs)
- Last 4 digits = season year

We extract just the year using `SUBSTR`.

In [5]:
# Verify season_id format
seasons = pd.read_sql_query("""
    SELECT DISTINCT season_id
    FROM game
    ORDER BY season_id
    LIMIT 10
""", conn)
print(seasons)

  season_id
0     12005
1     12006
2     12007
3     12008
4     12009
5     12010
6     12011
7     12012
8     12013
9     12014


## Step 4 — Build Stats One at a Time

We build each stat separately then join them all together at the end.
This keeps each query simple and easy to verify.

All queries join to `temp_valid_players` via the `games_played` base table to restrict to 2000+ draftees.

In [6]:
# Games played — count distinct games a player appeared in per season
# Joined to temp_valid_players to restrict to 2000+ draftees only
games_played = pd.read_sql_query("""
    SELECT
        p.player1_id AS player_id,
        p.player1_name AS player_name,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(DISTINCT p.game_id) AS games_played
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    JOIN temp_valid_players v ON p.player1_id = v.player_id
    WHERE p.player1_id IS NOT NULL
      AND p.player1_id != 0
    GROUP BY p.player1_id, season
""", conn)

print(f"Rows: {len(games_played)}")
games_played.head(10)

Rows: 6873


,player_id,player_name,season,games_played
0,101106,Andrew Bogut,2005,72
1,101106,Andrew Bogut,2006,60
2,101106,Andrew Bogut,2007,68
3,101106,Andrew Bogut,2008,35
4,101106,Andrew Bogut,2009,62
5,101106,Andrew Bogut,2010,58
6,101106,Andrew Bogut,2011,10
7,101106,Andrew Bogut,2012,11
8,101106,Andrew Bogut,2013,72
9,101106,Andrew Bogut,2014,82


In [7]:
# Field goals made — eventmsgtype = 1, player1 is the scorer
fgm = pd.read_sql_query("""
    SELECT
        p.player1_id AS player_id,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(*) AS fgm
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    WHERE p.eventmsgtype = 1
      AND p.player1_id IS NOT NULL
      AND p.player1_id != 0
    GROUP BY p.player1_id, season
""", conn)

print(f"Rows: {len(fgm)}")
fgm.head()

Rows: 13087


,player_id,season,fgm
0,100,1996,15
1,100,1997,2
2,100,1998,45
3,100,1999,21
4,1000,1996,151


In [8]:
# Rebounds — eventmsgtype = 4, player1 grabbed the rebound
rebounds = pd.read_sql_query("""
    SELECT
        p.player1_id AS player_id,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(*) AS rebounds
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    WHERE p.eventmsgtype = 4
      AND p.player1_id IS NOT NULL
      AND p.player1_id != 0
    GROUP BY p.player1_id, season
""", conn)

print(f"Rows: {len(rebounds)}")
rebounds.head()

Rows: 14062


,player_id,season,rebounds
0,100,1996,22
1,100,1997,3
2,100,1998,33
3,100,1999,19
4,1000,1996,197


In [9]:
# Assists — eventmsgtype = 1 (made shot), player2 is the assister
assists = pd.read_sql_query("""
    SELECT
        p.player2_id AS player_id,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(*) AS assists
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    WHERE p.eventmsgtype = 1
      AND p.player2_id IS NOT NULL
      AND p.player2_id != 0
    GROUP BY p.player2_id, season
""", conn)

print(f"Rows: {len(assists)}")
assists.head()

Rows: 12754


,player_id,season,assists
0,100,1996,9
1,100,1997,3
2,100,1998,20
3,100,1999,18
4,1000,1996,50


In [10]:
# Steals — eventmsgtype = 5 (turnover), player2 is the player who stole the ball
steals = pd.read_sql_query("""
    SELECT
        p.player2_id AS player_id,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(*) AS steals
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    WHERE p.eventmsgtype = 5
      AND p.player2_id IS NOT NULL
      AND p.player2_id != 0
    GROUP BY p.player2_id, season
""", conn)

print(f"Rows: {len(steals)}")
steals.head()

Rows: 12428


,player_id,season,steals
0,100,1996,3
1,100,1997,1
2,100,1998,4
3,100,1999,3
4,1000,1996,34


In [11]:
# Blocks — eventmsgtype = 2 (missed shot), player3 is the blocker
blocks = pd.read_sql_query("""
    SELECT
        p.player3_id AS player_id,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(*) AS blocks
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    WHERE p.eventmsgtype = 2
      AND p.player3_id IS NOT NULL
      AND p.player3_id != 0
    GROUP BY p.player3_id, season
""", conn)

print(f"Rows: {len(blocks)}")
blocks.head()

Rows: 11674


,player_id,season,blocks
0,100,1996,5
1,100,1998,3
2,100,1999,1
3,1000,1996,8
4,1000,1997,17


In [12]:
# Turnovers — eventmsgtype = 5, player1 committed the turnover
turnovers = pd.read_sql_query("""
    SELECT
        p.player1_id AS player_id,
        CAST(SUBSTR(g.season_id, 2, 4) AS INTEGER) AS season,
        COUNT(*) AS turnovers
    FROM play_by_play p
    JOIN game g ON p.game_id = g.game_id
    WHERE p.eventmsgtype = 5
      AND p.player1_id IS NOT NULL
      AND p.player1_id != 0
    GROUP BY p.player1_id, season
""", conn)

print(f"Rows: {len(turnovers)}")
turnovers.head()

Rows: 13726


,player_id,season,turnovers
0,100,1996,9
1,100,1997,4
2,100,1998,13
3,100,1999,3
4,1000,1996,79


## Step 5 — Join All Stats Together

We now have 7 separate DataFrames, one per stat. We join them all on `player_id` and `season`.

We use a left join from `games_played` as the base — if a player appeared in games, they should have a row even if they have zero assists or blocks.

In [13]:
# Start with games played as the base
player_season_stats = games_played.copy()

# Join each stat on player_id + season
for stat_df, stat_name in [
    (fgm, 'fgm'),
    (rebounds, 'rebounds'),
    (assists, 'assists'),
    (steals, 'steals'),
    (blocks, 'blocks'),
    (turnovers, 'turnovers')
]:
    player_season_stats = player_season_stats.merge(
        stat_df[['player_id', 'season', stat_name]],
        on=['player_id', 'season'],
        how='left'
    )

# Fill missing stats with 0 (player played but had none of that stat)
stat_cols = ['fgm', 'rebounds', 'assists', 'steals', 'blocks', 'turnovers']
player_season_stats[stat_cols] = player_season_stats[stat_cols].fillna(0).astype(int)

print(f"Total player-season rows: {len(player_season_stats)}")
player_season_stats.head(10)

Total player-season rows: 6873


,player_id,player_name,season,games_played,fgm,rebounds,assists,steals,blocks,turnovers
0,101106,Andrew Bogut,2005,72,288,500,175,44,65,112
1,101106,Andrew Bogut,2006,60,310,523,187,47,33,141
2,101106,Andrew Bogut,2007,68,402,657,175,56,119,156
3,101106,Andrew Bogut,2008,35,177,349,71,23,35,82
4,101106,Andrew Bogut,2009,62,423,629,115,35,162,117
5,101106,Andrew Bogut,2010,58,325,632,122,40,144,111
6,101106,Andrew Bogut,2011,10,46,81,25,11,18,19
7,101106,Andrew Bogut,2012,11,38,125,20,6,18,21
8,101106,Andrew Bogut,2013,72,247,709,121,49,124,102
9,101106,Andrew Bogut,2014,82,234,667,219,51,140,124


## Step 6 — Roll Up to Career Totals

Sum all seasons for each player to get career totals.

In [14]:
career_stats = player_season_stats.groupby(['player_id', 'player_name']).agg(
    seasons_played=('season', 'nunique'),
    games_played=('games_played', 'sum'),
    fgm=('fgm', 'sum'),
    rebounds=('rebounds', 'sum'),
    assists=('assists', 'sum'),
    steals=('steals', 'sum'),
    blocks=('blocks', 'sum'),
    turnovers=('turnovers', 'sum')
).reset_index()

print(f"Total players: {len(career_stats)}")
career_stats.sort_values('games_played', ascending=False).head(20)

Total players: 1198


,player_id,player_name,seasons_played,games_played,fgm,rebounds,assists,steals,blocks,turnovers
1105,2544,LeBron James,20,1463,14854,11516,10852,2292,1156,5267
1035,2225,Tony Parker,18,1271,7965,3490,6942,1088,101,2976
1160,2738,Andre Iguodala,19,1241,4816,5889,4962,1703,618,2119
912,2037,Jamal Crawford,20,1226,6162,2693,4128,1072,246,2231
2,101108,Chris Paul,18,1207,7774,5537,11422,2535,199,2979
1017,2207,Joe Johnson,18,1202,7297,4822,4614,1057,245,2315
1152,2730,Dwight Howard,18,1202,6824,14023,1610,1020,2142,3155
1148,2594,Kyle Korver,17,1199,3929,3409,1965,759,395,1245
1107,2546,Carmelo Anthony,19,1195,9600,7606,3217,1185,612,2890
1010,2200,Pau Gasol,18,1194,7716,11136,3826,600,1894,2528


## Step 7 — Sanity Check

Before saving, verify the numbers look reasonable against players we know.

In [15]:
check = ['LeBron James', 'Stephen Curry', 'Kevin Durant', 'Kobe Bryant']
career_stats[career_stats['player_name'].isin(check)]

,player_id,player_name,seasons_played,games_played,fgm,rebounds,assists,steals,blocks,turnovers
540,201142,Kevin Durant,15,1018,9718,7315,4402,1105,1143,3193
644,201939,Stephen Curry,14,904,7807,4488,5849,1482,228,2919
1105,2544,LeBron James,20,1463,14854,11516,10852,2292,1156,5267


## Step 8 — Save to CSV

In [16]:
OUTPUT_PATH = r"E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\player_career_stats.csv"
career_stats.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(career_stats)} players to {OUTPUT_PATH}")

# Verify Reggie Geary was excluded
reggie = career_stats[career_stats['player_name'] == 'Reggie Geary']
if len(reggie) == 0:
    print("CONFIRMED: Reggie Geary correctly excluded")
else:
    print("WARNING: Reggie Geary still in data!")

Saved 1198 players to E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\player_career_stats.csv
CONFIRMED: Reggie Geary correctly excluded
